# Datenexploration: FIT und Whoop

Ziel: die **tatsächlichen** Felder und Formate der Rohdaten inspizieren, bevor die
internen Datenmodelle festgelegt werden 
Die Rohdateien liegen in `data/private/` und sind nicht Teil des Repositories.

In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from pathlib import Path

import fitdecode
import polars as pl


REPO_ROOT = Path.cwd().parent  

FIT_PATH = next((REPO_ROOT / 'data' / 'private').glob('*.fit'))
CSV_PATH = next((REPO_ROOT / 'data' / 'private').glob('*/physiologische_zyklen.csv'))

print('FIT:', FIT_PATH.name)
print('CSV:', CSV_PATH.name)

FIT: 07-162x8minFTP.fit
CSV: physiologische_zyklen.csv


## 1. FIT-Datei

Eine Rad-Trainingseinheit (2×8 min FTP-Intervalle) mit Leistungsmesser.
Zuerst: welche Message-Typen und wie oft kommen sie vor?

In [2]:
msg_counts: Counter[str] = Counter()
fields_per_msg: dict[str, dict[str, str | None]] = defaultdict(dict)

with fitdecode.FitReader(str(FIT_PATH)) as reader:
    for frame in reader:
        if not isinstance(frame, fitdecode.FitDataMessage):
            continue
        msg_counts[frame.name] += 1
        for field in frame.fields:
            fields_per_msg[frame.name].setdefault(field.name, field.units)

for name, count in msg_counts.most_common():
    print(f'{name:20s} {count}')

record               4972
unknown_65280        310
device_info          63
unknown_65284        35
event                30
field_description    16
developer_data_id    14
power_zone           7
hr_zone              5
unknown_65281        2
file_id              1
sport                1
workout              1
lap                  1
session              1
activity             1


### Felder der `record`-Messages

Die eigentliche Zeitreihe (rund 1 Messpunkt pro Sekunde). Einheiten in Klammern.

In [3]:
for fname, units in fields_per_msg['record'].items():
    print(f'{fname:28s} ({units})')

timestamp                    (None)
heart_rate                   (bpm)
battery_soc                  (percent)
calories                     (kcal)
cadence                      (rpm)
power                        (watts)
left_pedal_smoothness        (percent)
right_pedal_smoothness       (percent)
left_torque_effectiveness    (percent)
right_torque_effectiveness   (percent)
temperature                  (C)
position_lat                 (semicircles)
position_long                (semicircles)
gps_accuracy                 (m)
distance                     (m)
enhanced_speed               (m/s)
speed                        (m/s)
enhanced_altitude            (m)
altitude                     (m)
grade                        (%)
ascent                       (m)
descent                      (m)


### Zusammenfassung der `session`-Message

GPS-Startkoordinaten werden aus Datenschutzgründen nicht ausgegeben.

In [4]:
with fitdecode.FitReader(str(FIT_PATH)) as reader:
    session = next(
        f
        for f in reader
        if isinstance(f, fitdecode.FitDataMessage) and f.name == 'session'
    )

for field in session.fields:
    if 'position' in field.name:
        continue
    print(f'{field.name:26s} {field.value!r} {field.units or ""}')

event                      'session' 
event_type                 'stop' 
timestamp                  datetime.datetime(2026, 7, 16, 15, 39, 18, tzinfo=datetime.timezone.utc) 
start_time                 datetime.datetime(2026, 7, 16, 14, 11, 39, tzinfo=datetime.timezone.utc) 
total_elapsed_time         5259.0 s
total_timer_time           4972.0 s
enhanced_avg_speed         6.911 m/s
avg_speed                  6.911 m/s
enhanced_max_speed         12.306 m/s
max_speed                  12.306 m/s
total_distance             34359.18 m
avg_cadence                78 rpm
max_cadence                110 rpm
min_heart_rate             98 bpm
avg_heart_rate             151 bpm
max_heart_rate             198 bpm
time_in_hr_zone            (1369.901, 2290.733, 379.456, 503.39, 409.539) s
avg_power                  129 watts
max_power                  625 watts
time_in_power_zone         (2237.667, 982.681, 406.409, 328.961, 391.879, 483.914, 126.07) s
total_work                 639616 J
enhanced_min_

### Welche Felder können fehlen?

In [5]:
core = [
    'timestamp', 'heart_rate', 'power', 'cadence',
    'distance', 'enhanced_speed', 'altitude', 'temperature',
]
none_counts = {k: 0 for k in core}
total = 0
sample: dict[str, object] | None = None

with fitdecode.FitReader(str(FIT_PATH)) as reader:
    for frame in reader:
        if not isinstance(frame, fitdecode.FitDataMessage) or frame.name != 'record':
            continue
        total += 1
        values = {f.name: f.value for f in frame.fields}
        for k in core:
            if values.get(k) is None:
                none_counts[k] += 1
        if total == 2500:
            sample = {k: values.get(k) for k in core}

print(f'Records gesamt: {total}\n')
print('Beispiel-Record (Nr. 2500):')
assert sample is not None
for k, v in sample.items():
    print(f'  {k:16s} {v!r}  ({type(v).__name__})')
print('\nFehlende Werte (None) je Feld:')
for k, v in none_counts.items():
    print(f'  {k:16s} {v}')

Records gesamt: 4972

Beispiel-Record (Nr. 2500):
  timestamp        datetime.datetime(2026, 7, 16, 14, 56, 19, tzinfo=datetime.timezone.utc)  (datetime)
  heart_rate       131  (int)
  power            0  (int)
  cadence          0  (int)
  distance         17760.8  (float)
  enhanced_speed   1.771  (float)
  altitude         40.60000000000002  (float)
  temperature      28  (int)

Fehlende Werte (None) je Feld:
  timestamp        0
  heart_rate       1
  power            2
  cadence          2
  distance         33
  enhanced_speed   36
  altitude         94
  temperature      2


### Fazit FIT

- Zeitstempel sind bereits **timezone-aware UTC** (`tzinfo=UTC`)
- Zeitreihe (`record`) fürs Modell: `timestamp`, `heart_rate`, `power`, `cadence`,
  `distance`, `enhanced_speed`, `altitude`. Felder können am Anfang/Ende fehlen → `Optional`.
- Die Einheit hat **genau eine** `session` und **einen** `lap` → Intervalle müssen per
  Sliding-Window erkannt werden (keine manuellen Runden vorhanden).
- GPS (`position_lat`/`position_long`) ist vorhanden, wird für die Kennzahlen aber nicht gebraucht.
- FTP/TSS/IF liefert Garmin bereits mit — später nützlich für die Trainingsbelastung.

## 2. Whoop: `physiologische_zyklen.csv`

In [6]:
df = pl.read_csv(CSV_PATH)
print('Zeilen, Spalten:', df.shape)
print()
for col, dtype in df.schema.items():
    print(f'{col:45s} {dtype}')

Zeilen, Spalten: (344, 26)

Startzeit des Zyklus                          String
Endzeit des Zyklus                            String
Zeitzone des Zyklus                           String
Erholungswert %                               Int64
Ruheherzfrequenz (Schläge pro Minute)         Int64
Herzfrequenzvariabilität (ms)                 Int64
Hauttemperatur (Celsius)                      Float64
Blutsauerstoff %                              Float64
Tagesbelastung                                Float64
Verbrannte Energie (cal)                      Int64
Max HF (Schläge pro Minute)                   Int64
Durchschnittliche HF (Schläge pro Minute)     Int64
Beginn des Schlafs                            String
Beginn des Aufwachens                         String
Schlafleistung %                              Int64
Atemfrequenz (Atemzüge/Min.)                  Float64
Schlafdauer (Min.)                            Int64
Dauer im Bett (Min.)                          Int64
Dauer des Leichtschlafs

### Schlüsselspalten

In [7]:
schluessel = [
    'Startzeit des Zyklus',
    'Zeitzone des Zyklus',
    'Erholungswert %',
    'Ruheherzfrequenz (Schläge pro Minute)',
    'Herzfrequenzvariabilität (ms)',
    'Hauttemperatur (Celsius)',
    'Atemfrequenz (Atemzüge/Min.)',
    'Blutsauerstoff %',
]
df.select(schluessel).head(5)

Startzeit des Zyklus,Zeitzone des Zyklus,Erholungswert %,Ruheherzfrequenz (Schläge pro Minute),Herzfrequenzvariabilität (ms),Hauttemperatur (Celsius),Atemfrequenz (Atemzüge/Min.),Blutsauerstoff %
str,str,i64,i64,i64,f64,f64,f64
"""2026-07-23 01:43:25""","""UTC+02:00""",73,57,98,33.04,14.9,96.27
"""2026-07-22 01:44:42""","""UTC+02:00""",74,56,98,33.7,14.7,97.65
"""2026-07-21 01:38:28""","""UTC+02:00""",59,53,88,33.82,14.9,96.55
"""2026-07-20 01:56:58""","""UTC+02:00""",65,53,91,33.48,14.7,94.82
"""2026-07-19 01:57:01""","""UTC+02:00""",58,55,88,33.61,15.0,97.22


### Fehlende Werte je Spalte

In [8]:
nulls = df.null_count()
for col in nulls.columns:
    print(f'{col:45s} {nulls[col][0]}')

Startzeit des Zyklus                          0
Endzeit des Zyklus                            1
Zeitzone des Zyklus                           0
Erholungswert %                               2
Ruheherzfrequenz (Schläge pro Minute)         2
Herzfrequenzvariabilität (ms)                 2
Hauttemperatur (Celsius)                      3
Blutsauerstoff %                              3
Tagesbelastung                                2
Verbrannte Energie (cal)                      2
Max HF (Schläge pro Minute)                   2
Durchschnittliche HF (Schläge pro Minute)     2
Beginn des Schlafs                            2
Beginn des Aufwachens                         2
Schlafleistung %                              2
Atemfrequenz (Atemzüge/Min.)                  2
Schlafdauer (Min.)                            2
Dauer im Bett (Min.)                          2
Dauer des Leichtschlafs (Min.)                2
Dauer des Tiefschlafs (Min.)                  2
Dauer des REM-Schlafs (Min.)            

### Fazit Whoop

- Eine Zeile pro Tag/Zyklus. Fürs Recovery-Modell zunächst nur die physiologischen
  Kennzahlen: Zyklus-Start, Zeitzone, Erholungswert, Ruheherzfrequenz, HRV,
  Hauttemperatur, Atemfrequenz, Blutsauerstoff.
- Zeit als lokale Startzeit + separate Zeitzonenspalte → beim Import nach UTC konvertieren.
- Fehlende Werte kommen vor → im Modell `Optional`.